# Phase 3 — NLP Symptom Extractor

### Install & setup

In [1]:
!pip install spacy fuzzywuzzy python-Levenshtein
!python -m spacy download en_core_web_sm

import spacy
import re
from fuzzywuzzy import fuzz, process

                                              0.0/15.4 MB ? eta -:--:--
                                              0.2/15.4 MB 3.5 MB/s eta 0:00:05
                                              0.3/15.4 MB 3.9 MB/s eta 0:00:04
     -                                        0.5/15.4 MB 3.1 MB/s eta 0:00:05
     -                                        0.6/15.4 MB 3.3 MB/s eta 0:00:05
     -                                        0.7/15.4 MB 3.2 MB/s eta 0:00:05
     --                                       1.0/15.4 MB 3.4 MB/s eta 0:00:05
     --                                       1.1/15.4 MB 3.4 MB/s eta 0:00:05
     ---                                      1.3/15.4 MB 3.4 MB/s eta 0:00:05
     ---                                      1.5/15.4 MB 3.6 MB/s eta 0:00:04
     ----                                     1.7/15.4 MB 3.5 MB/s eta 0:00:04
     ----                                     1.9/15.4 MB 3.8 MB/s eta 0:00:04
     -----                                    2.1/15.4 MB 3


[notice] A new release of pip is available: 23.1.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.1.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


                                              0.0/12.8 MB ? eta -:--:--
                                              0.1/12.8 MB 2.6 MB/s eta 0:00:05
                                              0.1/12.8 MB 2.0 MB/s eta 0:00:07
                                              0.3/12.8 MB 2.3 MB/s eta 0:00:06
     -                                        0.3/12.8 MB 2.2 MB/s eta 0:00:06
     -                                        0.4/12.8 MB 2.1 MB/s eta 0:00:06
     --                                       0.6/12.8 MB 2.4 MB/s eta 0:00:06
     --                                       0.7/12.8 MB 2.5 MB/s eta 0:00:05
     --                                       0.9/12.8 MB 2.8 MB/s eta 0:00:05
     ---                                      1.2/12.8 MB 2.8 MB/s eta 0:00:05
     ---                                      1.2/12.8 MB 2.8 MB/s eta 0:00:05
     ----                                     1.4/12.8 MB 2.8 MB/s eta 0:00:05
     ----                                     1.4/12.8 MB 2

In [2]:
nlp = spacy.load('en_core_web_sm')

### Load your symptom vocabulary from the trained model

In [ ]:
import pickle

with open('medisense_preprocessed.pkl', 'rb') as f:
    data = pickle.load(f)

feature_names = data['feature_names']   # 146 symptom columns
print(f"Total known symptoms: {len(feature_names)}")
print(feature_names[:15])

Total known symptoms: 146
['anxiety and nervousness', 'depression', 'shortness of breath', 'depressive or psychotic symptoms', 'sharp chest pain', 'dizziness', 'insomnia', 'abnormal involuntary movements', 'chest tightness', 'palpitations', 'irregular heartbeat', 'hoarse voice', 'sore throat', 'difficulty speaking', 'cough']


### Build symptom matcher (handles natural language → symptom columns)

In [ ]:
import re
from fuzzywuzzy import fuzz, process

symptom_lookup = {s.replace('_', ' ').lower(): s for s in feature_names}
symptom_phrases = list(symptom_lookup.keys())

# Synonym map built from ACTUAL 146-symptom vocabulary
SYNONYMS = {
    # General
    'tired': 'fatigue', 'exhausted': 'fatigue', 'no energy': 'fatigue',
    'high temperature': 'fever', 'temperature': 'fever',
    'cant sleep': 'insomnia', 'trouble sleeping': 'insomnia',
    'night sweats': 'sweating', 'sweats': 'sweating', 'sweating at night': 'sweating',
    'losing weight': 'weight gain',  # NOTE: only weight gain exists — flag mismatch, don't force match
    'aches all over': 'ache all over', 'body aches': 'ache all over',
    'feeling unwell': 'feeling ill', 'feeling sick': 'feeling ill',

    # Pain
    'stomach ache': 'sharp abdominal pain', 'stomach pain': 'sharp abdominal pain',
    'tummy ache': 'sharp abdominal pain', 'belly pain': 'sharp abdominal pain',
    'chest pain': 'sharp chest pain',
    'breathless': 'shortness of breath', 'cant breathe': 'difficulty breathing',
    'hard to breathe': 'difficulty breathing',
    'stiff joints': 'joint pain', 'morning stiffness': 'joint pain',
    'hands feel stiff': 'joint pain', 'joints feel stiff': 'joint pain',
    'achy joints': 'joint pain',

    # Urinary
    'burning while urinating': 'painful urination',
    'burning urination': 'painful urination', 'burning when i pee': 'painful urination',
    'peeing a lot': 'frequent urination', 'going to the bathroom a lot': 'frequent urination',
    'cant control urine': 'involuntary urination',
    'blood in pee': 'blood in urine', 'blood in my urine': 'blood in urine',

    # Respiratory
    'blood when coughing': 'coughing up sputum', 'coughing blood': 'coughing up sputum',
    'cough with blood': 'coughing up sputum', 'coughing up blood': 'coughing up sputum',
    'mucus': 'coughing up sputum', 'phlegm': 'coughing up sputum',
    'stuffy nose': 'nasal congestion', 'blocked nose': 'nasal congestion',
    'runny nose': 'coryza',

    # Skin
    'blisters': 'skin lesion', 'rash': 'skin rash', 'red rash': 'skin rash',
    'itchy skin': 'itching of skin', 'itchy': 'itching of skin',
    'dry skin': 'skin dryness, peeling, scaliness, or roughness',
    'peeling skin': 'skin dryness, peeling, scaliness, or roughness',
    'scaly skin': 'skin dryness, peeling, scaliness, or roughness',

    # GI
    'throwing up': 'vomiting', 'feeling nauseous': 'nausea', 'nauseous': 'nausea',
    'loose motions': 'diarrhea', 'loose stools': 'diarrhea',
    'cant poop': 'constipation', 'blocked up': 'constipation',
    'blood in poop': 'blood in stool', 'blood in stools': 'blood in stool',

    # Mental health
    'feeling anxious': 'anxiety and nervousness', 'nervous': 'anxiety and nervousness',
    'feeling sad': 'depression', 'low mood': 'depression',
    'mood swings': 'temper problems', 'irritable': 'temper problems',
    'angry': 'excessive anger', 'forgetful': 'disturbance of memory',
    'memory problems': 'disturbance of memory',

    # Eyes/Ears
    'blurry vision': 'diminished vision', 'blurred vision': 'diminished vision',
    'seeing double': 'double vision', 'red eyes': 'eye redness',
    'watery eyes': 'lacrimation', 'ear ache': 'ear pain',
    'ringing ears': 'ringing in ear', 'tinnitus': 'ringing in ear',

    # Heart
    'racing heart': 'increased heart rate', 'fast heartbeat': 'increased heart rate',
    'heart racing': 'palpitations', 'heart pounding': 'palpitations',
    'irregular heartbeat': 'irregular heartbeat', 'skipped heartbeat': 'irregular heartbeat',

    # Other
    'swollen legs': 'leg swelling', 'swollen feet': 'foot or toe swelling',
    'weak': 'weakness', 'numbness': 'loss of sensation',
    'tingling': 'paresthesia', 'pins and needles': 'paresthesia',
    'dizzy': 'dizziness', 'lightheaded': 'dizziness', 'fainted': 'fainting',
}

def generate_ngrams(text, max_n=5):
    words = text.split()
    ngrams = []
    for n in range(1, min(max_n, len(words)) + 1):
        for i in range(len(words) - n + 1):
            ngrams.append(' '.join(words[i:i+n]))
    return ngrams

In [ ]:
# Add more flexible variants to the synonym dict
SYNONYMS.update({
    'burning sensation while urinating': 'painful urination',
    'burning sensation when urinating': 'painful urination',
    'burns while urinating': 'painful urination',
    'pain while urinating': 'painful urination',
    'pain when urinating': 'painful urination',
    'hurts to pee': 'painful urination',
    'hurts when i pee': 'painful urination',
})

def extract_symptoms_from_text(text, threshold=80):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s']", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    found_symptoms = set()
    working_text = text
    for syn, canonical in SYNONYMS.items():
        if syn in working_text:
            working_text += ' | ' + canonical

    sorted_phrases = sorted(symptom_phrases, key=len, reverse=True)
    for phrase in sorted_phrases:
        if phrase in working_text:
            found_symptoms.add(symptom_lookup[phrase])

    ngrams = [g for g in generate_ngrams(working_text, max_n=4) if len(g) >= 4 and '|' not in g]
    for gram in ngrams:
        match, score = process.extractOne(gram, symptom_phrases, scorer=fuzz.token_sort_ratio)
        if score >= threshold:
            found_symptoms.add(symptom_lookup[match])

    # Dedup AFTER all matching is complete (substring + fuzzy)
    redundant_pairs = [
        ('back pain', 'low back pain'),
    ]
    for general, specific in redundant_pairs:
        general_col = symptom_lookup.get(general)
        specific_col = symptom_lookup.get(specific)
        if general_col in found_symptoms and specific_col in found_symptoms:
            found_symptoms.discard(general_col)

    return list(found_symptoms)

print(extract_symptoms_from_text("Burning sensation while urinating, lower back pain"))

['low back pain', 'painful urination']


### Build the prediction pipeline (text → symptoms → disease)

In [9]:
import numpy as np

def predict_disease_from_text(text, model, feature_names, label_encoder, top_n=3):
    extracted = extract_symptoms_from_text(text)

    if not extracted:
        return {
            'extracted_symptoms': [],
            'predictions': [],
            'message': 'No recognizable symptoms found. Please describe your symptoms more specifically.'
        }

    feature_vector = np.zeros(len(feature_names))
    for symptom in extracted:
        if symptom in feature_names:
            idx = feature_names.index(symptom)
            feature_vector[idx] = 1
    feature_vector = feature_vector.reshape(1, -1)

    probs = model.predict_proba(feature_vector)[0]
    top_indices = np.argsort(probs)[::-1][:top_n]

    predictions = [
        {
            'disease': label_encoder.inverse_transform([idx])[0],
            'confidence': round(float(probs[idx]) * 100, 2)
        }
        for idx in top_indices
    ]

    return {
        'extracted_symptoms': [s.replace('_', ' ') for s in extracted],
        'predictions': predictions,
        'message': 'success'
    }

# Final end-to-end test
for text in test_cases:
    result = predict_disease_from_text(text, data['xgb_model'], feature_names, data['label_encoder'], top_n=3)
    print(f"\n📝 {text}")
    print(f"   Symptoms: {result['extracted_symptoms']}")
    for p in result['predictions']:
        print(f"   → {p['disease']} ({p['confidence']}%)")


📝 I have joint pain and my hands feel stiff in the morning
   Symptoms: ['joint pain']
   → osteoporosis (59.27%)
   → sickle cell anemia (11.75%)
   → chronic knee pain (7.74%)

📝 Severe chest pain and shortness of breath, feeling dizzy
   Symptoms: ['dizziness', 'sharp chest pain', 'shortness of breath']
   → magnesium deficiency (30.56%)
   → paroxysmal ventricular tachycardia (12.22%)
   → chronic rheumatic fever (8.21%)

📝 Persistent cough with blood, night sweats, losing weight
   Symptoms: ['cough', 'weight gain', 'coughing up sputum', 'sweating']
   → obstructive sleep apnea (osa) (36.24%)
   → heart failure (21.65%)
   → acute sinusitis (15.88%)

📝 Burning sensation while urinating, lower back pain
   Symptoms: ['low back pain', 'painful urination']
   → headache after lumbar puncture (68.9%)
   → sciatica (4.28%)
   → epididymitis (4.22%)

📝 Itchy red rash all over my body with small blisters
   Symptoms: ['itching of skin', 'skin rash', 'skin lesion']
   → fungal infection 

### Save the NLP pipeline for later use

In [10]:
import pickle

nlp_artifacts = {
    'symptom_lookup': symptom_lookup,
    'symptom_phrases': symptom_phrases,
    'synonyms': SYNONYMS,
}

with open('nlp_artifacts.pkl', 'wb') as f:
    pickle.dump(nlp_artifacts, f)

print("NLP pipeline saved ✓")

NLP pipeline saved ✓
